# Day 1 - 배터리 데이터 로드

`archive/`의 `.mat` 파일(MATLAB v7.3 = HDF5)을 `h5py`로 읽습니다.

- `summary`: 셀별 cycle 단위 요약값 (IR, QDischarge, Tavg, chargetime 등) → 가볍고 빠름
- `cycles`: cycle 내부 시계열 (V, I, T, Qd, Qdlin 등) → 무거우므로 필요한 셀만 따로 로드

In [5]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

DATA_DIR = Path("archive")
mat_files = sorted(DATA_DIR.glob("*.mat"))
for p in mat_files:
    print(f"{p.name}  ({p.stat().st_size / 1e9:.2f} GB)")

2017-05-12_batchdata_updated_struct_errorcorrect.mat  (3.03 GB)
2018-02-20_batchdata_updated_struct_errorcorrect.mat  (2.02 GB)
2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat  (0.09 GB)
2018-04-12_batchdata_updated_struct_errorcorrect.mat  (3.24 GB)


## Batch 1 (2017-05-12) → `batch1` DataFrame

셀별 `summary`를 (cell_id, cycle) 단위 long format으로 합칩니다.

In [14]:
def mat_str(f, ref):
    """MATLAB char 배열(uint16) → str"""
    return "".join(chr(c) for c in f[ref][()].ravel())


def load_summary_df(path):
    dfs = []
    with h5py.File(path, "r") as f:
        batch = f["batch"]
        for i in range(batch["summary"].shape[0]):
            summary = f[batch["summary"][i, 0]]
            df = pd.DataFrame({k: summary[k][()].ravel() for k in summary.keys()})
            df.insert(0, "cell_id", i)
            df.insert(1, "policy", mat_str(f, batch["policy_readable"][i, 0]))
            df.insert(2, "cycle_life", f[batch["cycle_life"][i, 0]][()].item())
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


batch1 = load_summary_df(DATA_DIR / "2017-05-12_batchdata_updated_struct_errorcorrect.mat")
print(batch1.shape, "| cells:", batch1["cell_id"].nunique())
batch1.head()

(38811, 11) | cells: 46


,cell_id,policy,cycle_life,IR,QCharge,QDischarge,Tavg,Tmax,Tmin,chargetime,cycle
0,0,3.6C(80%)-3.6C,1190.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
1,0,3.6C(80%)-3.6C,1190.0,0.016742,1.071042,1.070689,31.875011,35.652016,29.566130,13.341250,2.0
2,0,3.6C(80%)-3.6C,1190.0,0.016724,1.071674,1.071900,31.931490,35.692978,29.604385,13.425777,3.0
3,0,3.6C(80%)-3.6C,1190.0,0.016681,1.072304,1.072510,31.932603,35.680588,29.744202,13.425167,4.0
4,0,3.6C(80%)-3.6C,1190.0,0.016662,1.072970,1.073174,31.959322,35.728691,29.644709,13.341442,5.0


## Batch 2 (2018-02-20), Batch 3 (2018-04-12)

In [15]:
batch2 = load_summary_df(DATA_DIR / "2018-02-20_batchdata_updated_struct_errorcorrect.mat")
batch3 = load_summary_df(DATA_DIR / "2018-04-12_batchdata_updated_struct_errorcorrect.mat")

for name, df in [("batch2", batch2), ("batch3", batch3)]:
    print(f"{name}: {df.shape} | cells: {df['cell_id'].nunique()}")

batch2: (26904, 11) | cells: 47
batch3: (51007, 11) | cells: 46


In [17]:
print(batch1.head())

   cell_id          policy  cycle_life        IR   QCharge  QDischarge  \
0        0  3.6C(80%)-3.6C      1190.0  0.000000  0.000000    0.000000   
1        0  3.6C(80%)-3.6C      1190.0  0.016742  1.071042    1.070689   
2        0  3.6C(80%)-3.6C      1190.0  0.016724  1.071674    1.071900   
3        0  3.6C(80%)-3.6C      1190.0  0.016681  1.072304    1.072510   
4        0  3.6C(80%)-3.6C      1190.0  0.016662  1.072970    1.073174   

        Tavg       Tmax       Tmin  chargetime  cycle  
0   0.000000   0.000000   0.000000    0.000000    1.0  
1  31.875011  35.652016  29.566130   13.341250    2.0  
2  31.931490  35.692978  29.604385   13.425777    3.0  
3  31.932603  35.680588  29.744202   13.425167    4.0  
4  31.959322  35.728691  29.644709   13.341442    5.0  
